# 料理・レシピ提案（初期コード）

条件を指定すると、登録済みのレシピから料理を最大3品提案し、指定した人数分の材料、作り方、目安の所要時間、概算の材料費を表示します。授業「計算科学基礎：2026年度　3Q（月3木3クラス）」のグループワークで、動かして試し、改善していくための出発点です。

## 1. はじめに

### 使い方

1. メニューの「ランタイム」→「すべてのセルを実行」を選びます。上から順に実行され、4節に提案が表示されます。
2. 4節のセルで入力条件 `conditions` の値を変え、そのセルをもう一度実行します（セルを選んで Shift + Enter）。
3. 5節の例と改善課題を参考に試し、条件と結果を記録して、グループで改善目標を決めます（記録用の表は、リポジトリの `projects/recipe-suggester/docs/experiments.md` にあります）。

- APIキー、追加パッケージのインストール、Google ドライブのマウント、ファイルのアップロードは不要です。インターネット上のデータも読み込みません。
- 6節の「生成AIによる提案」は任意の追加機能です。初期状態では無効で、APIを呼びません。

### 処理の流れ

```text
入力条件 conditions（4節）
  └ check_conditions …… 入力を確認し、食材名を食材IDに直す
登録済みレシピ RECIPES（2節）を1品ずつ調べる（suggest_recipes）
  ├ unmet_conditions …… 分類と「必ず使う食材」に合うかを調べる
  └ calculate_cost  …… 材料費を計算し、予算を超える料理を除く
ranking_score …… 「できれば使いたい食材」を多く使う順に並べる
show_suggestions …… 上位3品まで表示する（3品に満たなくても、条件に合わない料理で埋めない）
```

### このノートブックで使う主なPythonの要素

| 要素 | 使っている場所の例 |
| --- | --- |
| 変数 | `conditions`（入力条件）、`MAX_SUGGESTIONS`（表示する候補の数） |
| リスト | レシピの一覧 `RECIPES`、作り方の手順 `"steps"` |
| 辞書 | 単価表 `PRICE_TABLE`、1品分のレシピ、入力条件 `conditions` |
| 条件分岐 `if` | 分類が合うか（`unmet_conditions`）、予算内か（`suggest_recipes`） |
| 繰り返し `for` | レシピを1品ずつ調べる、材料費を足し合わせる（`calculate_cost`） |
| 関数 `def` | 入力の確認、絞り込み、費用計算、並べ替え、表示を、それぞれ関数に分けています |

**★ここを変えて試そう** と書かれた場所は、変更して試すことを想定した場所です。特に、入力条件（4節）、レシピの追加（2-4）、単価の変更（2-2）、順位付け（3-4）を変えてみましょう。

### 費用の前提

費用は、2節の単価表（授業用の仮の参考価格）と使用量から Python で計算した「使う量に相当する材料費」の概算です。調味料も含みます。実際の価格は地域・店舗・時期で変わり、1袋・1パック単位で買うときの支払額とも異なります。

## 2. データ

提案に使うデータです。すべてこのノートブックの中にあり、外部のファイルやインターネットは使いません。

### 2-1. 分類の選択肢と、入力条件の項目名

料理の種類、主な食材の区分、ジャンル、食事の場面で使える値です。入力条件では、これらに加えて「おまかせ」（指定しない）を使えます。レシピの分類も、ここにある値で書きます。

In [ ]:
# 分類の選択肢（「おまかせ」は「指定しない」という意味なので、ここには入れていません）
CHOICES = {
    "dish_type": ["麺類", "ご飯類", "スープ", "おかず"],  # 料理の種類
    "main_category": ["肉", "魚介", "野菜・豆類", "卵・乳製品"],  # 主な食材の区分
    "genre": ["和食", "中華", "洋食", "エスニック"],  # ジャンル
    "meal": ["朝食", "昼食", "夕食"],  # 食事の場面
}

# 入力条件の項目名（キー）と、画面に表示する日本語の名前
LABELS = {
    "dish_type": "料理の種類",
    "main_category": "主な食材の区分",
    "genre": "ジャンル",
    "meal": "食事の場面",
    "servings": "人数",
    "budget_per_person": "1人当たりの予算",
    "required_ingredients": "必ず使う食材",
    "wanted_ingredients": "できれば使いたい食材",
}

### 2-2. 単価表（授業用の仮の参考価格）

`PRICE_TABLE` は「食材ID → 表示名・単位・基準量・その基準量の価格」の辞書です。例えば `"pork"` は「豚こま切れ肉は 100 g 当たり 160 円」という意味で、150 g 使うなら 150 ÷ 100 × 160 = 240 円と計算します。

- 価格は授業用に決めた仮の値です。実際の店の価格を調べた値ではなく、地域・店舗・時期で変わります。
- 1つの食材IDには1つの単位だけを使います。レシピの使用量も、この単位で書きます（g と個のような単位の変換はしません）。
- 単価が分からない食材は `"price": None` にします。0円として計算せず、「費用算出不可」と表示します。
- 水（水道水）は費用がごくわずかなので、0円と決めています（単価が分からない場合とは区別しています）。

**★ここを変えて試そう**：価格を変えたり、新しい食材を追加したりしてみましょう。食材IDは英小文字と `_` で、ほかと重ならない名前にします。

In [ ]:
# 単価表：食材ID → {"name": 表示名, "unit": 単位, "base_amount": 基準量, "price": 基準量の参考価格（円）}
# 例："pork" は「豚こま切れ肉 100g 当たり 160円」。価格は授業用の仮の値です。
# ★ここを変えて試そう：価格を変える／新しい食材を追加する（単価が分からない食材は "price": None）
PRICE_TABLE = {
    # ご飯・麺
    "rice": {"name": "ご飯", "unit": "g", "base_amount": 100, "price": 40},  # 炊いたご飯
    "udon": {"name": "ゆでうどん", "unit": "玉", "base_amount": 1, "price": 50},
    "spaghetti": {"name": "スパゲッティ", "unit": "g", "base_amount": 100, "price": 40},  # 乾麺
    "pho": {"name": "フォー", "unit": "g", "base_amount": 100, "price": 120},  # 米粉の乾麺
    # 肉・魚介
    "pork": {"name": "豚こま切れ肉", "unit": "g", "base_amount": 100, "price": 160},
    "minced_pork": {"name": "豚ひき肉", "unit": "g", "base_amount": 100, "price": 150},
    "chicken": {"name": "鶏もも肉", "unit": "g", "base_amount": 100, "price": 140},
    "bacon": {"name": "ベーコン", "unit": "g", "base_amount": 100, "price": 250},
    "salmon": {"name": "生鮭", "unit": "切れ", "base_amount": 1, "price": 150},
    "prawn": {"name": "むきえび", "unit": "g", "base_amount": 100, "price": 300},
    "tinned_tuna": {"name": "ツナ缶", "unit": "缶", "base_amount": 1, "price": 120},
    # 卵・乳製品・豆腐
    "egg": {"name": "卵", "unit": "個", "base_amount": 1, "price": 25},
    "milk": {"name": "牛乳", "unit": "ml", "base_amount": 100, "price": 25},
    "butter": {"name": "バター", "unit": "g", "base_amount": 10, "price": 20},
    "tofu": {"name": "豆腐", "unit": "丁", "base_amount": 1, "price": 90},
    # 野菜など
    "onion": {"name": "玉ねぎ", "unit": "個", "base_amount": 1, "price": 50},
    "carrot": {"name": "にんじん", "unit": "本", "base_amount": 1, "price": 50},
    "potato": {"name": "じゃがいも", "unit": "個", "base_amount": 1, "price": 40},
    "cabbage": {"name": "キャベツ", "unit": "g", "base_amount": 100, "price": 30},
    "bean_sprouts": {"name": "もやし", "unit": "g", "base_amount": 100, "price": 20},
    "spring_onion": {"name": "長ねぎ", "unit": "本", "base_amount": 1, "price": 70},
    "green_pepper": {"name": "ピーマン", "unit": "個", "base_amount": 1, "price": 30},
    "tomato": {"name": "トマト", "unit": "個", "base_amount": 1, "price": 80},
    "tinned_tomatoes": {"name": "カットトマト缶", "unit": "缶", "base_amount": 1, "price": 120},
    "ginger": {"name": "しょうが", "unit": "g", "base_amount": 10, "price": 15},
    "garlic": {"name": "にんにく", "unit": "かけ", "base_amount": 1, "price": 15},
    "nori": {"name": "焼きのり", "unit": "枚", "base_amount": 1, "price": 15},
    "lemon_juice": {"name": "レモン汁", "unit": "ml", "base_amount": 10, "price": 10},
    # 調味料・粉・水
    "soy_sauce": {"name": "しょうゆ", "unit": "ml", "base_amount": 100, "price": 30},
    "sugar": {"name": "砂糖", "unit": "g", "base_amount": 100, "price": 25},
    "salt": {"name": "塩", "unit": "g", "base_amount": 10, "price": 1},
    "black_pepper": {"name": "こしょう", "unit": "g", "base_amount": 1, "price": 5},
    "oil": {"name": "サラダ油", "unit": "ml", "base_amount": 100, "price": 40},
    "sesame_oil": {"name": "ごま油", "unit": "ml", "base_amount": 10, "price": 15},
    "miso": {"name": "みそ", "unit": "g", "base_amount": 100, "price": 60},
    "dashi_powder": {"name": "和風だしの素", "unit": "g", "base_amount": 10, "price": 30},
    "chicken_stock": {"name": "鶏がらスープの素", "unit": "g", "base_amount": 10, "price": 25},
    "consomme": {"name": "コンソメ", "unit": "g", "base_amount": 10, "price": 25},
    "ketchup": {"name": "トマトケチャップ", "unit": "g", "base_amount": 100, "price": 50},
    "curry_roux": {"name": "カレールウ", "unit": "g", "base_amount": 100, "price": 150},
    "fish_sauce": {"name": "ナンプラー", "unit": "ml", "base_amount": 10, "price": 20},
    "doubanjiang": {"name": "豆板醤", "unit": "g", "base_amount": 10, "price": 30},
    "potato_starch": {"name": "片栗粉", "unit": "g", "base_amount": 10, "price": 4},
    "flour": {"name": "小麦粉", "unit": "g", "base_amount": 100, "price": 30},
    "water": {"name": "水", "unit": "ml", "base_amount": 100, "price": 0},  # 水道水はごくわずかなので0円とする
}

### 2-3. 表記ゆれ辞書

入力した食材名は、まず単価表の表示名（`"玉ねぎ"` など）と比べ、見つからなければこの辞書で食材IDに直します。「たまご」「豚肉」のような別の書き方を登録しておくと、入力しやすくなります。

**★ここを変えて試そう**：よく使う書き方を追加してみましょう（右側は単価表にある食材IDにします）。

In [ ]:
# 表記ゆれ辞書：入力に使える別の書き方 → 食材ID
INGREDIENT_ALIASES = {
    "ごはん": "rice", "白ご飯": "rice",
    "うどん": "udon",
    "パスタ": "spaghetti", "スパゲティ": "spaghetti",
    "豚肉": "pork", "豚こま": "pork", "ぶた肉": "pork",
    "ひき肉": "minced_pork", "豚ミンチ": "minced_pork",
    "鶏肉": "chicken", "とり肉": "chicken", "鶏もも": "chicken",
    "鮭": "salmon", "さけ": "salmon", "サケ": "salmon", "サーモン": "salmon",
    "えび": "prawn", "エビ": "prawn", "海老": "prawn",
    "ツナ": "tinned_tuna",
    "たまご": "egg", "玉子": "egg", "タマゴ": "egg",
    "とうふ": "tofu",
    "たまねぎ": "onion", "タマネギ": "onion", "玉葱": "onion",
    "人参": "carrot", "ニンジン": "carrot",
    "ジャガイモ": "potato", "じゃが芋": "potato",
    "ねぎ": "spring_onion", "ネギ": "spring_onion", "長ネギ": "spring_onion",
    "トマト缶": "tinned_tomatoes",
    "生姜": "ginger", "ショウガ": "ginger",
    "ニンニク": "garlic",
    "のり": "nori", "海苔": "nori",
    "醤油": "soy_sauce", "しょう油": "soy_sauce",
    "ケチャップ": "ketchup",
    "カレールー": "curry_roux",
}

### 2-4. レシピ（20品）

`RECIPES` は、1品を1つの辞書で表したレシピのリストです。料理と手順は、この授業のために用意した簡単な例で、特定のレシピサイトの文章ではありません。

| キー | 内容 |
| --- | --- |
| `"id"` | レシピの識別子（ほかと重ならない文字列） |
| `"name"` | 料理名 |
| `"dish_types"` | 料理の種類のリスト（複数可） |
| `"main_categories"` | 主な食材の区分のリスト（複数可） |
| `"genre"` | ジャンル（1つ） |
| `"meals"` | 食事の場面のリスト（複数可） |
| `"base_servings"` | 基準人数（下の材料の量が何人分か） |
| `"ingredients"` | 材料の辞書 `{食材ID: 基準人数分の使用量}`。単位は単価表の `"unit"` |
| `"steps"` | 作り方の手順のリスト。人数を変えても使えるように、分量は書きません |
| `"minutes"` | 目安の所要時間（分）。人数を変えても同じ値を使います |

分類は、データに書いたタグだけで判定します（AIなどによる推測は使いません）。ゆでるための湯は材料に含めていません。

**★ここを変えて試そう**：最後のレシピをまねて、リストの最後に新しい料理を追加してみましょう。追加したら、3-6 の `check_recipe_data()` で書き方を確認できます。

In [ ]:
# レシピのリスト（材料の量は "base_servings" 人分。単位は単価表の "unit"）
# ★ここを変えて試そう：最後のレシピをまねて、リストの最後に新しい料理を追加する
RECIPES = [
    {
        "id": "R01",
        "name": "親子丼",
        "dish_types": ["ご飯類"],
        "main_categories": ["肉", "卵・乳製品"],
        "genre": "和食",
        "meals": ["昼食", "夕食"],
        "base_servings": 2,
        "ingredients": {"rice": 400, "chicken": 200, "egg": 3, "onion": 0.5,
                        "soy_sauce": 30, "sugar": 12, "dashi_powder": 2, "water": 150},
        "steps": [
            "鶏もも肉をひと口大に切り、玉ねぎを薄切りにする。",
            "小さめのフライパンに水、和風だしの素、しょうゆ、砂糖を入れて煮立てる。",
            "鶏もも肉と玉ねぎを加え、ふたをして中火で5分ほど煮る。",
            "卵を溶きほぐして回し入れ、ふたをして半熟になるまで1分ほど煮る。",
            "器にご飯を盛り、上にのせる。",
        ],
        "minutes": 20,
    },
    {
        "id": "R02",
        "name": "鮭おにぎり",
        "dish_types": ["ご飯類"],
        "main_categories": ["魚介"],
        "genre": "和食",
        "meals": ["朝食", "昼食"],
        "base_servings": 2,
        "ingredients": {"rice": 400, "salmon": 1, "salt": 2, "nori": 1},
        "steps": [
            "生鮭に塩の半量をふり、フライパンかグリルで両面をしっかり焼く。",
            "焼いた鮭の皮と骨を取り除き、身をほぐす。",
            "ご飯にほぐした身を混ぜ、手に残りの塩をつけて、食べやすい大きさに握る。",
            "焼きのりを切って、おにぎりに巻く。",
        ],
        "minutes": 20,
    },
    {
        "id": "R03",
        "name": "卵チャーハン",
        "dish_types": ["ご飯類"],
        "main_categories": ["卵・乳製品"],
        "genre": "中華",
        "meals": ["昼食", "夕食"],
        "base_servings": 2,
        "ingredients": {"rice": 400, "egg": 2, "spring_onion": 0.5, "oil": 15,
                        "chicken_stock": 3, "salt": 1, "black_pepper": 0.2, "soy_sauce": 5},
        "steps": [
            "長ねぎをみじん切りにし、卵を溶きほぐす。",
            "フライパンにサラダ油を強火で熱し、溶き卵を流し入れて、すぐにご飯を加えてほぐしながら炒める。",
            "長ねぎ、鶏がらスープの素、塩、こしょうを加えて炒め合わせる。",
            "しょうゆをフライパンの縁から回し入れ、全体を混ぜる。",
        ],
        "minutes": 15,
    },
    {
        "id": "R04",
        "name": "カレーライス",
        "dish_types": ["ご飯類"],
        "main_categories": ["肉", "野菜・豆類"],
        "genre": "洋食",
        "meals": ["夕食"],
        "base_servings": 4,
        "ingredients": {"rice": 800, "pork": 250, "onion": 2, "carrot": 1, "potato": 2,
                        "oil": 15, "water": 700, "curry_roux": 100},
        "steps": [
            "玉ねぎはくし形切り、にんじんとじゃがいもはひと口大に切る。豚こま切れ肉は大きければ切る。",
            "鍋にサラダ油を熱し、豚こま切れ肉と玉ねぎを炒める。",
            "にんじんとじゃがいもを加えて軽く炒め、水を加えて煮立てる。あくを取り、ふたをして弱火で15分ほど煮る。",
            "火を止めてカレールウを溶かし入れ、とろみがつくまで弱火で5分ほど煮る。",
            "器にご飯を盛り、カレーをかける。",
        ],
        "minutes": 45,
    },
    {
        "id": "R05",
        "name": "ガパオ風ライス",
        "dish_types": ["ご飯類"],
        "main_categories": ["肉"],
        "genre": "エスニック",
        "meals": ["昼食", "夕食"],
        "base_servings": 2,
        "ingredients": {"rice": 400, "minced_pork": 200, "onion": 0.5, "green_pepper": 2, "garlic": 1,
                        "egg": 2, "oil": 15, "fish_sauce": 10, "soy_sauce": 5, "sugar": 4},
        "steps": [
            "玉ねぎとピーマンを1cm角に切り、にんにくをみじん切りにする。",
            "フライパンにサラダ油の半量を熱して卵を割り入れ、目玉焼きを作って取り出す。",
            "同じフライパンに残りのサラダ油とにんにくを入れて熱し、豚ひき肉を炒める。",
            "玉ねぎとピーマンを加えて炒め、ナンプラー、しょうゆ、砂糖で味付けする。",
            "器にご飯を盛り、炒めたものと目玉焼きをのせる。",
        ],
        "minutes": 20,
    },
    {
        "id": "R06",
        "name": "焼きうどん",
        "dish_types": ["麺類"],
        "main_categories": ["肉"],
        "genre": "和食",
        "meals": ["昼食", "夕食"],
        "base_servings": 2,
        "ingredients": {"udon": 2, "pork": 150, "cabbage": 150, "carrot": 0.3,
                        "oil": 10, "dashi_powder": 2, "soy_sauce": 20},
        "steps": [
            "キャベツはざく切り、にんじんは細切りにし、豚こま切れ肉は食べやすい大きさに切る。",
            "ゆでうどんは電子レンジで軽く温めて、ほぐしておく。",
            "フライパンにサラダ油を熱して豚こま切れ肉を炒め、色が変わったらキャベツとにんじんを加えて炒める。",
            "ゆでうどんを加えて炒め合わせ、和風だしの素としょうゆで味を調える。",
        ],
        "minutes": 15,
    },
    {
        "id": "R07",
        "name": "月見うどん",
        "dish_types": ["麺類"],
        "main_categories": ["卵・乳製品"],
        "genre": "和食",
        "meals": ["朝食", "昼食"],
        "base_servings": 1,
        "ingredients": {"udon": 1, "egg": 1, "spring_onion": 0.2, "water": 300,
                        "dashi_powder": 2, "soy_sauce": 15, "sugar": 3},
        "steps": [
            "長ねぎを小口切りにする。",
            "鍋に水と和風だしの素を入れて沸かし、しょうゆと砂糖で味を調えてつゆを作る。",
            "ゆでうどんを加えて温め、中央に卵を割り入れて、白身が固まるまで弱火で2分ほど煮る。",
            "器に盛り、長ねぎをのせる。",
        ],
        "minutes": 10,
    },
    {
        "id": "R08",
        "name": "ナポリタン",
        "dish_types": ["麺類"],
        "main_categories": ["肉"],
        "genre": "洋食",
        "meals": ["昼食"],
        "base_servings": 2,
        "ingredients": {"spaghetti": 200, "bacon": 60, "onion": 0.5, "green_pepper": 2,
                        "oil": 10, "ketchup": 80, "salt": 1, "black_pepper": 0.2},
        "steps": [
            "玉ねぎを薄切り、ピーマンを細切り、ベーコンを1cm幅に切る。",
            "スパゲッティを袋の表示どおりにゆでる。",
            "フライパンにサラダ油を熱し、ベーコン、玉ねぎ、ピーマンを炒める。",
            "トマトケチャップを加えて軽く炒め、ゆでたスパゲッティを加えて混ぜ、塩とこしょうで味を調える。",
        ],
        "minutes": 20,
    },
    {
        "id": "R09",
        "name": "ツナとトマトのスパゲッティ",
        "dish_types": ["麺類"],
        "main_categories": ["魚介"],
        "genre": "洋食",
        "meals": ["昼食", "夕食"],
        "base_servings": 2,
        "ingredients": {"spaghetti": 200, "tinned_tuna": 1, "tinned_tomatoes": 1, "onion": 0.5,
                        "garlic": 1, "oil": 15, "salt": 2, "black_pepper": 0.2},
        "steps": [
            "玉ねぎとにんにくをみじん切りにする。",
            "スパゲッティを袋の表示どおりにゆでる。",
            "フライパンにサラダ油とにんにくを入れて弱火で熱し、香りが出たら玉ねぎを炒める。",
            "カットトマト缶と、油を切ったツナ缶を加えて5分ほど煮詰め、塩とこしょうで味を調える。",
            "ゆでたスパゲッティを加え、ソースとよく混ぜる。",
        ],
        "minutes": 20,
    },
    {
        "id": "R10",
        "name": "鶏肉のフォー風スープ麺",
        "dish_types": ["麺類", "スープ"],
        "main_categories": ["肉"],
        "genre": "エスニック",
        "meals": ["昼食"],
        "base_servings": 2,
        "ingredients": {"pho": 100, "chicken": 150, "bean_sprouts": 100, "spring_onion": 0.3,
                        "water": 700, "chicken_stock": 6, "fish_sauce": 10, "lemon_juice": 10},
        "steps": [
            "鶏もも肉を薄いそぎ切りにし、長ねぎを斜め薄切りにする。",
            "鍋に水と鶏がらスープの素を入れて沸かし、鶏もも肉を加えて火が通るまで5分ほど煮る。",
            "ナンプラーで味を調え、もやしを加えてさっと煮る。",
            "フォーを袋の表示どおりにゆでて器に入れ、スープと具を注ぐ。",
            "長ねぎをのせ、食べる直前にレモン汁をかける。",
        ],
        "minutes": 20,
    },
    {
        "id": "R11",
        "name": "豆腐とねぎのみそ汁",
        "dish_types": ["スープ"],
        "main_categories": ["野菜・豆類"],
        "genre": "和食",
        "meals": ["朝食", "夕食"],
        "base_servings": 2,
        "ingredients": {"tofu": 0.5, "spring_onion": 0.3, "water": 400, "dashi_powder": 2, "miso": 30},
        "steps": [
            "豆腐を1.5cm角に切り、長ねぎを小口切りにする。",
            "鍋に水と和風だしの素を入れて沸かし、豆腐を加えて2分ほど煮る。",
            "火を弱めてみそを溶き入れ、長ねぎを加えて、煮立つ直前に火を止める。",
        ],
        "minutes": 10,
    },
    {
        "id": "R12",
        "name": "中華風卵スープ",
        "dish_types": ["スープ"],
        "main_categories": ["卵・乳製品"],
        "genre": "中華",
        "meals": ["朝食", "昼食", "夕食"],
        "base_servings": 2,
        "ingredients": {"egg": 1, "spring_onion": 0.3, "water": 400, "chicken_stock": 5,
                        "salt": 0.5, "potato_starch": 3, "sesame_oil": 2},
        "steps": [
            "長ねぎを斜め薄切りにし、卵を溶きほぐす。片栗粉は、分量のうち少しの水で溶いておく。",
            "鍋に残りの水と鶏がらスープの素を入れて沸かし、塩で味を調える。",
            "水で溶いた片栗粉を加えてとろみをつけ、溶き卵を細く回し入れる。",
            "長ねぎを加えて火を止め、ごま油をたらす。",
        ],
        "minutes": 10,
    },
    {
        "id": "R13",
        "name": "ミネストローネ",
        "dish_types": ["スープ"],
        "main_categories": ["野菜・豆類"],
        "genre": "洋食",
        "meals": ["昼食", "夕食"],
        "base_servings": 4,
        "ingredients": {"onion": 1, "carrot": 0.5, "potato": 1, "cabbage": 150, "bacon": 60,
                        "oil": 10, "tinned_tomatoes": 1, "water": 600, "consomme": 8, "salt": 2},
        "steps": [
            "玉ねぎ、にんじん、じゃがいも、キャベツを1cm角に切り、ベーコンを1cm幅に切る。",
            "鍋にサラダ油を熱し、ベーコン、玉ねぎ、にんじんを炒める。",
            "じゃがいも、キャベツ、カットトマト缶、水、コンソメを加え、ふたをして弱火で15分ほど煮る。",
            "野菜がやわらかくなったら、塩で味を調える。",
        ],
        "minutes": 30,
    },
    {
        "id": "R14",
        "name": "えびのトムヤム風スープ",
        "dish_types": ["スープ"],
        "main_categories": ["魚介"],
        "genre": "エスニック",
        "meals": ["夕食"],
        "base_servings": 2,
        "ingredients": {"prawn": 120, "tomato": 1, "onion": 0.5, "water": 500, "chicken_stock": 5,
                        "doubanjiang": 3, "fish_sauce": 10, "sugar": 3, "lemon_juice": 15},
        "steps": [
            "トマトをくし形切り、玉ねぎを薄切りにする。むきえびは背わたがあれば取る。",
            "鍋に水と鶏がらスープの素を入れて沸かし、玉ねぎを加えて3分ほど煮る。",
            "むきえびとトマトを加え、えびの色が変わるまで煮る。",
            "豆板醤、ナンプラー、砂糖で味付けし、火を止めてからレモン汁を加える。",
        ],
        "minutes": 20,
    },
    {
        "id": "R15",
        "name": "豚肉のしょうが焼き",
        "dish_types": ["おかず"],
        "main_categories": ["肉"],
        "genre": "和食",
        "meals": ["昼食", "夕食"],
        "base_servings": 2,
        "ingredients": {"pork": 200, "onion": 0.5, "ginger": 10, "soy_sauce": 20,
                        "sugar": 6, "oil": 10, "cabbage": 100},
        "steps": [
            "しょうがをすりおろし、しょうゆと砂糖を混ぜてたれを作る。玉ねぎは薄切りにする。",
            "キャベツを千切りにして、器に盛っておく。",
            "フライパンにサラダ油を熱し、豚こま切れ肉と玉ねぎを炒める。",
            "肉に火が通ったらたれを加え、汁けが少なくなるまで炒め合わせて、キャベツの横に盛る。",
        ],
        "minutes": 15,
    },
    {
        "id": "R16",
        "name": "麻婆豆腐",
        "dish_types": ["おかず"],
        "main_categories": ["肉", "野菜・豆類"],
        "genre": "中華",
        "meals": ["夕食"],
        "base_servings": 2,
        "ingredients": {"tofu": 1, "minced_pork": 100, "spring_onion": 0.3, "garlic": 1, "ginger": 5,
                        "oil": 10, "doubanjiang": 5, "water": 150, "chicken_stock": 3,
                        "soy_sauce": 15, "sugar": 3, "potato_starch": 5},
        "steps": [
            "豆腐を2cm角に切る。長ねぎ、にんにく、しょうがをみじん切りにする。片栗粉は、分量のうち少しの水で溶いておく。",
            "フライパンにサラダ油、にんにく、しょうが、豆板醤を入れて弱火で炒め、香りが出たら豚ひき肉を加えて炒める。",
            "残りの水、鶏がらスープの素、しょうゆ、砂糖を加えて煮立て、豆腐を入れて3分ほど煮る。",
            "水で溶いた片栗粉を回し入れてとろみをつけ、長ねぎを加えて混ぜる。",
        ],
        "minutes": 20,
    },
    {
        "id": "R17",
        "name": "鮭のムニエル",
        "dish_types": ["おかず"],
        "main_categories": ["魚介"],
        "genre": "洋食",
        "meals": ["夕食"],
        "base_servings": 2,
        "ingredients": {"salmon": 2, "salt": 1, "black_pepper": 0.2, "flour": 10,
                        "butter": 10, "lemon_juice": 10},
        "steps": [
            "生鮭の表面をキッチンペーパーでふき、塩とこしょうをふって5分ほど置く。",
            "生鮭の両面に小麦粉を薄くまぶす。",
            "フライパンにバターを溶かし、生鮭を中火で両面こんがりと焼く。",
            "器に盛り、レモン汁をかける。",
        ],
        "minutes": 15,
    },
    {
        "id": "R18",
        "name": "卵焼き",
        "dish_types": ["おかず"],
        "main_categories": ["卵・乳製品"],
        "genre": "和食",
        "meals": ["朝食", "昼食"],
        "base_servings": 2,
        "ingredients": {"egg": 3, "sugar": 6, "soy_sauce": 3, "oil": 5},
        "steps": [
            "卵を溶きほぐし、砂糖としょうゆを加えてよく混ぜる。",
            "フライパンにサラダ油を薄くひき、卵液の3分の1ほどを流し入れる。",
            "半熟になったら奥から手前に巻き、残りの卵液も2回に分けて同じように焼いて巻く。",
            "少し冷ましてから、食べやすい大きさに切る。",
        ],
        "minutes": 10,
    },
    {
        "id": "R19",
        "name": "スクランブルエッグ",
        "dish_types": ["おかず"],
        "main_categories": ["卵・乳製品"],
        "genre": "洋食",
        "meals": ["朝食"],
        "base_servings": 1,
        "ingredients": {"egg": 2, "milk": 30, "salt": 0.5, "black_pepper": 0.1, "butter": 10},
        "steps": [
            "ボウルに卵を割りほぐし、牛乳、塩、こしょうを加えて混ぜる。",
            "フライパンにバターを入れて弱火で溶かし、卵液を流し入れる。",
            "ゴムべらなどで大きく混ぜ、半熟のうちに火を止めて器に盛る。",
        ],
        "minutes": 5,
    },
    {
        "id": "R20",
        "name": "もやしとキャベツの野菜炒め",
        "dish_types": ["おかず"],
        "main_categories": ["野菜・豆類"],
        "genre": "中華",
        "meals": ["昼食", "夕食"],
        "base_servings": 2,
        "ingredients": {"bean_sprouts": 200, "cabbage": 150, "carrot": 0.3, "green_pepper": 1,
                        "oil": 10, "chicken_stock": 3, "salt": 0.5, "black_pepper": 0.2},
        "steps": [
            "キャベツはざく切り、にんじんは短冊切り、ピーマンは細切りにする。",
            "フライパンにサラダ油を強火で熱し、にんじんを炒める。",
            "キャベツ、ピーマン、もやしを加えて手早く炒める。",
            "鶏がらスープの素、塩、こしょうで味を調える。",
        ],
        "minutes": 10,
    },
]

## 3. 関数

入力の確認、絞り込み、費用計算、並べ替え、表示を、役割ごとの関数に分けています。まずは次の関数から読んでみましょう。

| 関数 | 役割 |
| --- | --- |
| `check_conditions`（3-1） | 入力条件を確認し、食材名を食材IDに直す |
| `unmet_conditions`（3-2） | レシピが満たしていない条件を調べる |
| `calculate_cost`（3-3） | 指定人数分の量と材料費を計算する |
| `ranking_score`（3-4） | 並べ替えに使う点数を決める（★ここを変えて試そう） |
| `suggest_recipes`（3-4） | 絞り込み → 費用計算 → 予算の判定 → 並べ替え をまとめて行う |
| `show_suggestions`（3-5） | 入力の確認から表示までを行う（4節で使う） |
| `check_recipe_data`（3-6） | 単価表とレシピの書き方を確認する |

3-1〜3-5 のセルは関数を定義するだけなので、実行しても何も表示されません。

### 3-1. 入力の確認と整形

入力条件 `conditions` を確認し、検索に使う形 `request` に整えます。

- 分類は、選択肢か「おまかせ」かを確かめます。前後の空白は取り除きます。
- 人数は1以上の整数、予算は `None`（制限なし）か0より大きい数かを確かめます。
- 食材名は、空白・空の名前・重複を取り除いてから、単価表の名前と表記ゆれ辞書で食材IDに直します。
- 単価表にない「必ず使う食材」は、黙って除かずにエラーとして知らせます。単価表にない「できれば使いたい食材」は、順位付けに使えないことを知らせます。

In [ ]:
MAX_SUGGESTIONS = 3  # 表示する候補の最大数


def find_food_id(name):
    """食材名から食材IDを探す。単価表の表示名 → 表記ゆれ辞書の順に探し、見つからなければ None を返す。"""
    for food_id in PRICE_TABLE:
        if PRICE_TABLE[food_id]["name"] == name:
            return food_id
    if name in INGREDIENT_ALIASES:
        return INGREDIENT_ALIASES[name]
    return None


def food_name(food_id):
    """食材IDから表示名を返す（単価表にないIDは、そのまま文字列にして返す）。"""
    if food_id in PRICE_TABLE:
        return PRICE_TABLE[food_id]["name"]
    return str(food_id)


def is_text_list(value):
    """value が「文字列だけが入ったリスト」なら True を返す（空のリストも True）。"""
    if not isinstance(value, list):
        return False
    for item in value:
        if not isinstance(item, str):
            return False
    return True


def names_to_ids(names):
    """食材名のリストを食材IDのリストに直す。空白、空の名前、重複は取り除く。
    戻り値は (食材IDのリスト, 見つからなかった名前のリスト)。"""
    food_ids = []
    unknown_names = []
    for name in names:
        name = name.replace(" ", "").replace("　", "")  # 半角と全角の空白を取り除く
        if name == "":
            continue
        food_id = find_food_id(name)
        if food_id is None:
            if name not in unknown_names:
                unknown_names.append(name)
        elif food_id not in food_ids:
            food_ids.append(food_id)
    return food_ids, unknown_names


def check_conditions(conditions):
    """入力条件を確認し、検索に使う形の辞書 request に整える。
    戻り値は (request, errors, notes)。
      errors … 直さないと検索できない問題のリスト（1つでもあれば request は None）
      notes  … 検索はできるが、知っておいてほしいことのリスト
    """
    errors = []
    notes = []
    request = {}

    # 使われない項目名（綴りの間違いなど）
    for key in conditions:
        if key not in LABELS:
            notes.append("項目「" + str(key) + "」は使われていません。項目名の綴りを確認してください。")

    # 分類：選択肢のどれかか「おまかせ」
    for key in CHOICES:
        value = conditions.get(key)
        if isinstance(value, str):
            value = value.strip()
        if value != "おまかせ" and value not in CHOICES[key]:
            errors.append(LABELS[key] + "（" + key + "）は「" + "」「".join(CHOICES[key])
                          + "」「おまかせ」のどれかにしてください。今の値：" + repr(value))
        request[key] = value

    # 人数：1以上の整数（True と False は数として扱わない）
    servings = conditions.get("servings")
    if isinstance(servings, bool) or not isinstance(servings, int):
        errors.append('人数（servings）は 2 のような整数で書いてください（"2" や 2.5 は使えません）。今の値：'
                      + repr(servings))
    elif servings < 1:
        errors.append("人数（servings）は1以上にしてください。今の値：" + repr(servings))
    request["servings"] = servings

    # 予算：None（制限なし）か、0より大きい数
    budget = conditions.get("budget_per_person")
    if budget is not None:
        if isinstance(budget, bool) or not isinstance(budget, (int, float)):
            errors.append("1人当たりの予算（budget_per_person）は 300 のような数で書いてください。"
                          + "制限しないときは None にします。今の値：" + repr(budget))
        elif budget <= 0:
            errors.append("1人当たりの予算（budget_per_person）は0より大きい数にしてください。"
                          + "制限しないときは None にします。今の値：" + repr(budget))
    request["budget_per_person"] = budget

    # 必ず使う食材：単価表にない名前は、黙って除かずにエラーにする
    required = conditions.get("required_ingredients")
    if not is_text_list(required):
        errors.append('必ず使う食材（required_ingredients）は ["卵", "長ねぎ"] のような文字列のリストで書いてください。'
                      + "指定しないときは [] にします。今の値：" + repr(required))
        required = []
    required_ids, unknown_names = names_to_ids(required)
    for name in unknown_names:
        errors.append("必ず使う食材「" + name + "」は単価表にありません。"
                      + "show_food_names() で表示される名前で書くか、単価表に追加してください。")
    request["required_ids"] = required_ids

    # できれば使いたい食材：単価表にない名前は、順位付けに使えないことを知らせる
    wanted = conditions.get("wanted_ingredients")
    if not is_text_list(wanted):
        errors.append('できれば使いたい食材（wanted_ingredients）は ["長ねぎ"] のような文字列のリストで書いてください。'
                      + "指定しないときは [] にします。今の値：" + repr(wanted))
        wanted = []
    wanted_ids, unknown_names = names_to_ids(wanted)
    for name in unknown_names:
        notes.append("できれば使いたい食材「" + name + "」は単価表にないため、順位付けに使いません。")
    request["wanted_ids"] = []
    for food_id in wanted_ids:
        if food_id in required_ids:
            notes.append("「" + food_name(food_id) + "」は必ず使う食材にも入っているため、できれば使いたい食材からは外しました。")
        else:
            request["wanted_ids"].append(food_id)

    if len(errors) > 0:
        return None, errors, notes
    return request, errors, notes

### 3-2. 条件に合うかを調べる

レシピの分類と材料を、整えた条件 `request` と比べます。「おまかせ」の項目は調べません。合わない条件があれば、その理由を文のリストで返します（空のリストなら、すべての条件に合っています）。この関数は、6節で生成AIの提案を確かめるときにも使います。

In [ ]:
def unmet_conditions(recipe, request):
    """レシピが満たしていない条件を、理由の文のリストで返す（空のリストなら、すべて満たしている）。"""
    reasons = []
    if request["dish_type"] != "おまかせ" and request["dish_type"] not in recipe["dish_types"]:
        reasons.append("料理の種類が「" + request["dish_type"] + "」ではありません")
    if request["main_category"] != "おまかせ" and request["main_category"] not in recipe["main_categories"]:
        reasons.append("主な食材の区分が「" + request["main_category"] + "」ではありません")
    if request["genre"] != "おまかせ" and request["genre"] != recipe["genre"]:
        reasons.append("ジャンルが「" + request["genre"] + "」ではありません")
    if request["meal"] != "おまかせ" and request["meal"] not in recipe["meals"]:
        reasons.append("食事の場面に「" + request["meal"] + "」が含まれていません")
    for food_id in request["required_ids"]:
        if food_id not in recipe["ingredients"]:
            reasons.append("必ず使う食材「" + food_name(food_id) + "」を使っていません")
    return reasons

### 3-3. 材料の量と費用の計算

基準人数の材料の量を、指定人数に合わせて比例計算し（例：2人分のレシピを3人分にするなら1.5倍）、単価表から材料ごとの費用を計算します。

- この単純なモデルでは、合計費用は人数に比例して増え、1人当たりの費用は人数を変えても変わりません。実際には、少しだけ使う調味料や、まとめて作る料理などで、比例しないこともあります。
- 単価が分からない材料があるときは、合計を `None`（費用算出不可）にします。0円として足すことはしません。
- 円の表示は `format_yen` で1円未満を四捨五入してそろえます。予算との比較には、丸める前の値を使います。そのため、材料ごとの表示を足した値と、合計の表示が1円ほどずれることがあります。

In [ ]:
def calculate_cost(recipe, servings):
    """指定人数分の材料の量と、使う量に相当する材料費を計算する。
    戻り値の辞書：
      "items"      … 材料ごとの {"food_id", "name", "amount", "unit", "cost"} のリスト
      "total"      … 合計の費用（円）。単価が分からない材料があれば None
      "per_person" … 1人当たりの費用（円）。合計が None なら None
      "missing"    … 単価が分からない材料の名前のリスト
    """
    scale = servings / recipe["base_servings"]  # 基準人数の何倍の量を作るか
    items = []
    total = 0
    missing = []
    for food_id, base_amount in recipe["ingredients"].items():
        amount = base_amount * scale
        if food_id in PRICE_TABLE:
            unit = PRICE_TABLE[food_id]["unit"]
            price = PRICE_TABLE[food_id]["price"]
        else:
            unit = ""
            price = None
        if price is None:
            cost = None  # 単価が分からない材料は、0円にせず「費用算出不可」にする
            missing.append(food_name(food_id))
        else:
            cost = amount / PRICE_TABLE[food_id]["base_amount"] * price  # 基準量の何倍か × 基準量の価格
            total = total + cost
        items.append({"food_id": food_id, "name": food_name(food_id), "amount": amount, "unit": unit, "cost": cost})

    if len(missing) > 0:
        return {"items": items, "total": None, "per_person": None, "missing": missing}
    return {"items": items, "total": total, "per_person": total / servings, "missing": missing}


def format_amount(amount):
    """数量を表示用の文字列にする（1未満は小数第2位まで、それ以外は小数第1位まで。末尾の .0 は付けない）。"""
    if amount < 1:
        text = str(round(amount, 2))
    else:
        text = str(round(amount, 1))
    if text.endswith(".0"):
        text = text[:-2]
    return text


def format_yen(value):
    """金額を「約1,234円」の形にする（1円未満は四捨五入）。表示のためだけに丸めます。"""
    if value is None:
        return "費用算出不可"
    return f"約{int(value + 0.5):,}円"  # :, は3桁ごとにカンマを入れる書き方


def price_text(food_id):
    """単価表の1行を「100g当たり160円」のような文字列にする。"""
    info = PRICE_TABLE[food_id]
    if info["price"] is None:
        return "単価なし"
    return format_amount(info["base_amount"]) + info["unit"] + "当たり" + format_amount(info["price"]) + "円"

### 3-4. 候補を探して並べる

`suggest_recipes` は、すべてのレシピを1品ずつ調べて、次の順に処理します。

1. 分類と「必ず使う食材」に合わない料理を除く（`unmet_conditions`）。
2. 材料費を計算し、予算を超える料理を除く（`calculate_cost`）。費用を計算できない料理は、予算内と判定しません。
3. `ranking_score` の点数が大きい順に並べる。

今の `ranking_score` は「できれば使いたい食材をいくつ使うか」だけで点数を決めます。同じ点数の料理は、`sorted` の性質で `RECIPES` に登録した順に並びます（乱数は使わないので、同じ条件なら毎回同じ結果になります）。

**★ここを変えて試そう**：`ranking_score` の計算を変えると、並び順が変わります。

In [ ]:
def count_wanted(recipe, wanted_ids):
    """できれば使いたい食材のうち、レシピで使うものの数を数える。"""
    count = 0
    for food_id in wanted_ids:
        if food_id in recipe["ingredients"]:
            count = count + 1
    return count


def ranking_score(candidate):
    """並べ替えに使う点数を返す（大きいほど上位）。
    ★ここを変えて試そう：今は「できれば使いたい食材を使う数」だけで決めています。
    candidate["recipe"]（レシピ）や candidate["cost"]（費用の計算結果）も使えます。"""
    return candidate["wanted_count"]


def suggest_recipes(request):
    """条件に合う料理を探して並べる。
    戻り値の辞書：
      "ranked"       … 条件に合う料理（並べ替え済み）のリスト
      "top"          … そのうち上位 MAX_SUGGESTIONS 品
      "over_budget"  … 分類と必ず使う食材には合うが、予算を超えた料理
      "unknown_cost" … 費用を計算できず、予算内か判定できなかった料理
    """
    candidates = []
    over_budget = []
    unknown_cost = []
    budget = request["budget_per_person"]
    for recipe in RECIPES:
        # 1. 分類と「必ず使う食材」で絞り込む
        if len(unmet_conditions(recipe, request)) > 0:
            continue
        # 2. 材料費を計算し、予算を超える料理を除く（比較は丸める前の値で行う）
        cost = calculate_cost(recipe, request["servings"])
        if budget is not None:
            if cost["per_person"] is None:
                unknown_cost.append(recipe)
                continue
            if cost["per_person"] > budget:
                over_budget.append({"recipe": recipe, "per_person": cost["per_person"]})
                continue
        # 3. 候補に加える
        candidates.append({
            "recipe": recipe,
            "cost": cost,
            "wanted_count": count_wanted(recipe, request["wanted_ids"]),
        })

    # 4. 点数の大きい順に並べる（sorted は、同じ点数なら元の順番を保つ）
    ranked = sorted(candidates, key=ranking_score, reverse=True)
    return {
        "ranked": ranked,
        "top": ranked[:MAX_SUGGESTIONS],
        "over_budget": over_budget,
        "unknown_cost": unknown_cost,
    }

### 3-5. 表示

入力条件、候補の料理、候補がないときの見直し方を表示する関数です。`show_suggestions` は、入力の確認から表示までをまとめて行います。

In [ ]:
PRICE_NOTE = (
    "※ 費用は、授業用の仮の参考単価から計算した「使う量に相当する材料費」の概算です（調味料を含みます）。\n"
    "  実際の価格は地域・店舗・時期で変わり、1袋・1パック単位で買うときの支払額とも異なります。"
)


def names_text(food_ids):
    """食材IDのリストを「卵、長ねぎ」のような文字列にする（空のリストなら「なし」）。"""
    if len(food_ids) == 0:
        return "なし"
    names = []
    for food_id in food_ids:
        names.append(food_name(food_id))
    return "、".join(names)


def show_food_names():
    """入力に使える食材の名前（単価表の表示名）を表示する。"""
    names = []
    for food_id in PRICE_TABLE:
        names.append(PRICE_TABLE[food_id]["name"])
    print("使える食材の名前（" + str(len(names)) + "品目）：")
    print("、".join(names))
    print("ほかに、表記ゆれ辞書 INGREDIENT_ALIASES にある書き方（たまご、豚肉、ねぎ など）も使えます。")


def show_request(request):
    """整えた入力条件を表示する。"""
    print("【入力条件】")
    for key in CHOICES:
        print("  " + LABELS[key] + "：" + request[key])
    print("  人数：" + str(request["servings"]) + "人分")
    if request["budget_per_person"] is None:
        print("  1人当たりの予算：制限なし")
    else:
        print("  1人当たりの予算：" + format_amount(request["budget_per_person"]) + "円まで")
    print("  必ず使う食材：" + names_text(request["required_ids"]))
    print("  できれば使いたい食材：" + names_text(request["wanted_ids"]))


def show_candidate(number, candidate, request, label):
    """候補1品分を表示する。label には「登録済みレシピからの提案」などの説明を入れる。"""
    recipe = candidate["recipe"]
    cost = candidate["cost"]
    print("=" * 50)
    print("候補" + str(number) + "：" + recipe["name"] + "（" + label + "）")
    print("分類：料理の種類「" + "・".join(recipe["dish_types"]) + "」、主な食材「" + "・".join(recipe["main_categories"])
          + "」、ジャンル「" + recipe["genre"] + "」、食事の場面「" + "・".join(recipe["meals"]) + "」")
    print("目安の所要時間：約" + format_amount(recipe["minutes"]) + "分（人数を変えても同じ目安です）")
    print()

    print("■ 材料（" + str(request["servings"]) + "人分）と概算の材料費")
    for item in cost["items"]:
        if item["food_id"] in PRICE_TABLE:
            basis = price_text(item["food_id"])
        else:
            basis = "単価表にない食材"
        print("・" + item["name"] + " " + format_amount(item["amount"]) + item["unit"]
              + "：" + format_yen(item["cost"]) + "（" + basis + "）")
    if cost["total"] is None:
        print("合計：費用算出不可（単価が分からない材料：" + "、".join(cost["missing"]) + "）")
    else:
        print("合計：" + format_yen(cost["total"]) + "（1人当たり " + format_yen(cost["per_person"]) + "）")
    print()

    print("■ 作り方")
    step_number = 1
    for step in recipe["steps"]:
        print(str(step_number) + ". " + step)
        step_number = step_number + 1
    print()

    if len(request["wanted_ids"]) == 0:
        print("■ できれば使いたい食材：指定なし")
    else:
        used = []
        unused = []
        for food_id in request["wanted_ids"]:
            if food_id in recipe["ingredients"]:
                used.append(food_id)
            else:
                unused.append(food_id)
        print("■ できれば使いたい食材")
        print("使う：" + names_text(used))
        print("使わない：" + names_text(unused))
    print()


def show_no_candidates(result, request):
    """候補がないときに、その旨と条件の見直し方を表示する（条件を自動で緩めることはしない）。"""
    print("条件に合う料理がありません。")
    print("条件は自動では緩めていません。次のように見直してみてください。")
    hints = []
    if len(result["over_budget"]) > 0:
        cheapest = None
        for item in result["over_budget"]:
            if cheapest is None or item["per_person"] < cheapest:
                cheapest = item["per_person"]
        hints.append("1人当たりの予算を上げる（ほかの条件には合うが予算を超えた料理が" + str(len(result["over_budget"]))
                     + "品あります。最も安いもので1人当たり" + format_yen(cheapest) + "です）")
    if len(result["unknown_cost"]) > 0:
        hints.append("単価表に単価を追加する（費用を計算できず、予算内か判定できなかった料理が"
                     + str(len(result["unknown_cost"])) + "品あります）")
    for key in CHOICES:
        if request[key] != "おまかせ":
            hints.append(LABELS[key] + "の「" + request[key] + "」を「おまかせ」にする")
    if len(request["required_ids"]) > 0:
        hints.append("必ず使う食材（" + names_text(request["required_ids"]) + "）を減らすか、「できれば使いたい食材」に移す")
    if len(hints) == 0:
        hints.append("レシピのデータ RECIPES が空になっていないか確認する")
    for hint in hints:
        print("・" + hint)


def prepare_request(conditions):
    """入力条件を確認して表示する。直すところがあれば、その内容を表示して None を返す。"""
    request, errors, notes = check_conditions(conditions)
    if len(errors) > 0:
        print("【入力条件を直してください】")
        for message in errors:
            print("・" + message)
    else:
        show_request(request)
    for note in notes:
        print("※ " + note)
    return request


def show_basic_result(request):
    """登録済みレシピから料理を探し、結果を表示する（基本モード）。"""
    result = suggest_recipes(request)
    print()
    if len(result["top"]) == 0:
        show_no_candidates(result, request)
        return
    if len(result["ranked"]) > len(result["top"]):
        print("条件に合う料理は" + str(len(result["ranked"])) + "品です。上位" + str(len(result["top"])) + "品を表示します。")
    else:
        print("条件に合う料理は" + str(len(result["ranked"])) + "品です。")
    print(PRICE_NOTE)
    number = 1
    for candidate in result["top"]:
        show_candidate(number, candidate, request, "登録済みレシピからの提案")
        number = number + 1


def show_suggestions(conditions):
    """入力条件を確認し、登録済みレシピからの提案を表示する（4節と5節で使います）。"""
    request = prepare_request(conditions)
    if request is not None:
        show_basic_result(request)

### 3-6. データの確認

単価表、表記ゆれ辞書、レシピの書き方に問題がないかを確かめます。分類が選択肢にない、食材IDが単価表にない、量が0以下、などを見つけます。`find_recipe_problems` は、6節で生成AIの提案を確かめるときにも使います。

**★レシピや単価を追加・変更したら、このセルを実行して確認しよう。**

In [ ]:
def list_tag_problems(tags, label, options):
    """分類タグのリストを確かめ、問題を文のリストで返す。"""
    if not isinstance(tags, list) or len(tags) == 0:
        return [label + "は、選択肢の値を1つ以上入れたリストにしてください（今の値：" + repr(tags) + "）"]
    problems = []
    for tag in tags:
        if tag not in options:
            problems.append(label + "の" + repr(tag) + "は選択肢（" + "、".join(options) + "）にありません")
    return problems


def find_recipe_problems(recipe):
    """1品分のレシピの書き方を確かめ、問題を文のリストで返す（空のリストなら問題なし）。"""
    problems = []
    name = recipe.get("name")
    if not isinstance(name, str) or name.strip() == "":
        problems.append("料理名（name）がありません")

    problems = problems + list_tag_problems(recipe.get("dish_types"), "料理の種類（dish_types）", CHOICES["dish_type"])
    problems = problems + list_tag_problems(recipe.get("main_categories"), "主な食材の区分（main_categories）",
                                            CHOICES["main_category"])
    problems = problems + list_tag_problems(recipe.get("meals"), "食事の場面（meals）", CHOICES["meal"])
    if recipe.get("genre") not in CHOICES["genre"]:
        problems.append("ジャンル（genre）は「" + "」「".join(CHOICES["genre"]) + "」のどれかにしてください（今の値："
                        + repr(recipe.get("genre")) + "）")

    servings = recipe.get("base_servings")
    if isinstance(servings, bool) or not isinstance(servings, int) or servings < 1:
        problems.append("基準人数（base_servings）は1以上の整数にしてください（今の値：" + repr(servings) + "）")

    ingredients = recipe.get("ingredients")
    if not isinstance(ingredients, dict) or len(ingredients) == 0:
        problems.append("材料（ingredients）は {食材ID: 使用量} の辞書にして、1つ以上書いてください")
    else:
        for food_id, amount in ingredients.items():
            if food_id not in PRICE_TABLE:
                problems.append("食材ID「" + str(food_id) + "」は単価表にありません")
            if isinstance(amount, bool) or not isinstance(amount, (int, float)):
                problems.append("「" + food_name(food_id) + "」の量が数ではありません（今の値：" + repr(amount) + "）")
            elif amount <= 0:
                problems.append("「" + food_name(food_id) + "」の量が0以下です（今の値：" + repr(amount) + "）")

    steps = recipe.get("steps")
    if not is_text_list(steps) or len(steps) == 0:
        problems.append("作り方（steps）は文字列のリストにして、1つ以上書いてください")

    minutes = recipe.get("minutes")
    if isinstance(minutes, bool) or not isinstance(minutes, (int, float)) or minutes <= 0:
        problems.append("所要時間（minutes）は0より大きい数にしてください（今の値：" + repr(minutes) + "）")
    return problems


def check_recipe_data():
    """単価表・表記ゆれ辞書・レシピをまとめて確認し、結果を表示する。"""
    problems = []
    names = []
    for food_id in PRICE_TABLE:
        info = PRICE_TABLE[food_id]
        base = info.get("base_amount")
        price = info.get("price")
        if not isinstance(info.get("name"), str) or not isinstance(info.get("unit"), str):
            problems.append("単価表「" + food_id + "」：name と unit は文字列にしてください")
        elif info["name"] in names:
            problems.append("単価表「" + food_id + "」：表示名「" + info["name"] + "」がほかの食材と重なっています")
        else:
            names.append(info["name"])
        if isinstance(base, bool) or not isinstance(base, (int, float)) or base <= 0:
            problems.append("単価表「" + food_id + "」：base_amount は0より大きい数にしてください")
        if price is not None and (isinstance(price, bool) or not isinstance(price, (int, float)) or price < 0):
            problems.append("単価表「" + food_id + "」：price は0以上の数か None にしてください")

    for alias in INGREDIENT_ALIASES:
        if INGREDIENT_ALIASES[alias] not in PRICE_TABLE:
            problems.append("表記ゆれ辞書「" + alias + "」：食材ID「" + str(INGREDIENT_ALIASES[alias]) + "」が単価表にありません")

    recipe_ids = []
    for recipe in RECIPES:
        title = "レシピ「" + str(recipe.get("id")) + " " + str(recipe.get("name")) + "」："
        if not isinstance(recipe.get("id"), str) or recipe.get("id") == "":
            problems.append(title + "id がありません")
        elif recipe["id"] in recipe_ids:
            problems.append(title + "id がほかのレシピと重なっています")
        recipe_ids.append(recipe.get("id"))
        for problem in find_recipe_problems(recipe):
            problems.append(title + problem)

    if len(problems) == 0:
        print("単価表" + str(len(PRICE_TABLE)) + "品目、表記ゆれ" + str(len(INGREDIENT_ALIASES)) + "件、レシピ"
              + str(len(RECIPES)) + "品を確認しました。問題は見つかりませんでした。")
    else:
        print("データに直してほしいところが" + str(len(problems)) + "件あります。")
        for problem in problems:
            print("・" + problem)


check_recipe_data()

## 4. 入力条件を決めて、提案を表示する

下の `conditions` の値を変えて、そのセルを実行すると、提案が表示されます。

| 項目（キー） | 意味 | 書き方の例 |
| --- | --- | --- |
| `"dish_type"` | 料理の種類 | `"麺類"`、`"ご飯類"`、`"スープ"`、`"おかず"`、`"おまかせ"` |
| `"main_category"` | 主な食材の区分 | `"肉"`、`"魚介"`、`"野菜・豆類"`、`"卵・乳製品"`、`"おまかせ"` |
| `"genre"` | ジャンル | `"和食"`、`"中華"`、`"洋食"`、`"エスニック"`、`"おまかせ"` |
| `"meal"` | 食事の場面 | `"朝食"`、`"昼食"`、`"夕食"`、`"おまかせ"` |
| `"servings"` | 人数（1以上の整数） | `1`、`2`、`4` |
| `"budget_per_person"` | 1人当たりの材料費の上限（円）。制限しないときは `None` | `300`、`None` |
| `"required_ingredients"` | 必ず使う食材。すべてを使う料理だけを探します。指定しないときは `[]` | `["卵"]`、`["豚肉", "キャベツ"]` |
| `"wanted_ingredients"` | できれば使いたい食材。多く使う料理ほど上位にします。指定しないときは `[]` | `["長ねぎ", "にんじん"]` |

- 「料理の種類」（麺類・ご飯類・スープ・おかず）と「主な食材の区分」（肉・魚介など）は別の条件です。
- 「使いたい食材」は、家にあることや費用が0円であることを意味しません。使う材料は、すべて材料費に含めます。
- 条件に合う料理がないときは、条件を自動で緩めずに「条件に合う料理がありません」と表示し、見直し方を示します。

まず、入力に使える食材の名前を確認しておきましょう。

In [ ]:
show_food_names()

In [ ]:
# ★ここを変えて試そう：入力条件（値を変えて、このセルを実行します）
conditions = {
    "dish_type": "おまかせ",  # 料理の種類：麺類／ご飯類／スープ／おかず／おまかせ
    "main_category": "おまかせ",  # 主な食材の区分：肉／魚介／野菜・豆類／卵・乳製品／おまかせ
    "genre": "和食",  # ジャンル：和食／中華／洋食／エスニック／おまかせ
    "meal": "夕食",  # 食事の場面：朝食／昼食／夕食／おまかせ
    "servings": 2,  # 人数：1以上の整数
    "budget_per_person": 300,  # 1人当たりの材料費の上限（円）。制限しないときは None
    "required_ingredients": [],  # 必ず使う食材（例：["卵"]）
    "wanted_ingredients": ["長ねぎ", "にんじん"],  # できれば使いたい食材
}

show_suggestions(conditions)

## 5. 条件を変えて試す例と、改善課題

### 5-1. 条件を変えて試す例

それぞれのセルは、4節の `conditions` とは別の条件で試します。値を変えて、結果がどう変わるかを確かめてみましょう。

In [ ]:
# 例1：朝食向けの料理を、卵を使うものを優先して探す（予算は制限なし）
example1 = {
    "dish_type": "おまかせ",
    "main_category": "おまかせ",
    "genre": "おまかせ",
    "meal": "朝食",
    "servings": 1,
    "budget_per_person": None,
    "required_ingredients": [],
    "wanted_ingredients": ["卵"],
}
show_suggestions(example1)

In [ ]:
# 例2：「必ず使う食材」と「できれば使いたい食材」を両方指定する
example2 = {
    "dish_type": "おまかせ",
    "main_category": "おまかせ",
    "genre": "おまかせ",
    "meal": "おまかせ",
    "servings": 2,
    "budget_per_person": 250,
    "required_ingredients": ["卵"],
    "wanted_ingredients": ["長ねぎ", "ご飯"],
}
show_suggestions(example2)

In [ ]:
# 例3：人数を変えると、材料の量と費用はどう変わる？（RECIPES の最初の料理で確かめる）
recipe = RECIPES[0]
print(recipe["name"] + "（基準人数：" + str(recipe["base_servings"]) + "人分）")
for servings in [1, 2, 3, 4]:
    cost = calculate_cost(recipe, servings)
    first_item = cost["items"][0]  # 1つ目の材料
    print(str(servings) + "人分：" + first_item["name"] + " " + format_amount(first_item["amount"]) + first_item["unit"]
          + "、合計 " + format_yen(cost["total"]) + "、1人当たり " + format_yen(cost["per_person"]))

In [ ]:
# 例4：候補が見つからない例（エスニックの朝食は登録されていない／予算が極端に低い）
example4 = {
    "dish_type": "おまかせ",
    "main_category": "おまかせ",
    "genre": "エスニック",
    "meal": "朝食",
    "servings": 2,
    "budget_per_person": None,
    "required_ingredients": [],
    "wanted_ingredients": [],
}
show_suggestions(example4)

print()
print("-" * 50)
example4["genre"] = "おまかせ"
example4["meal"] = "おまかせ"
example4["budget_per_person"] = 20
show_suggestions(example4)

In [ ]:
# 例5：入力の間違いがあると、直し方が表示される
example5 = {
    "dish_type": "麺",  # 「麺類」の書き間違い
    "main_category": "おまかせ",
    "genre": "和食",
    "meal": "夕食",
    "servings": 0,  # 1以上でないといけない
    "budget_per_person": "500",  # 文字列ではなく数で書く
    "required_ingredients": ["パクチー"],  # 単価表にない食材
    "wanted_ingredients": [],
}
show_suggestions(example5)

In [ ]:
# 例6：単価表にない食材を使うと「費用算出不可」になる（この料理は RECIPES には追加していません）
test_recipe = {
    "id": "TEST1",
    "name": "しいたけの卵とじ（確認用）",
    "dish_types": ["おかず"],
    "main_categories": ["卵・乳製品"],
    "genre": "和食",
    "meals": ["夕食"],
    "base_servings": 2,
    "ingredients": {"egg": 2, "shiitake": 4, "soy_sauce": 10},  # "shiitake" は単価表にない
    "steps": ["しいたけを薄切りにする。", "しょうゆを加えて煮て、溶いた卵でとじる。"],
    "minutes": 10,
}
cost = calculate_cost(test_recipe, 2)
for item in cost["items"]:
    print(item["name"] + "：" + format_yen(item["cost"]))
print("合計：" + format_yen(cost["total"]) + "（単価が分からない材料：" + "、".join(cost["missing"]) + "）")
for problem in find_recipe_problems(test_recipe):
    print("データの確認：" + problem)

### 5-2. 改善課題

改善には、次の3種類があります。何を変えたのか、なぜ変えたのかを説明できるようにしましょう。

- **Pythonの処理を変える**：関数の中身を変えます。同じ入力に対して、毎回同じように結果が変わるので、改善前と同じ条件で比べて確かめられます。
- **データを変える**：単価表やレシピを追加・変更します。
- **AIへの指示を変える**（6節）：生成AIへの頼み方を変えます。結果は実行のたびに変わることがあり、正しいとは限らないので、Python の確認処理で確かめます。

| No. | 種類 | 課題 | 主に変更する場所 |
| --- | --- | --- | --- |
| 1 | Pythonの処理 | 同じ点数のときは登録順になるため、`RECIPES` の前の方の料理が提案されやすくなっています。1人当たりの費用が安い順や、所要時間が短い順も考えて並べてみよう | `ranking_score`（3-4） |
| 2 | Pythonの処理 | 朝食を選んだときは、所要時間が15分以内の料理だけを候補にしてみよう（「所要時間の上限」という入力条件を追加してもよい） | `unmet_conditions`（3-2）、`conditions`（4節）、`LABELS`（2-1）、`check_conditions`（3-1） |
| 3 | データ | レシピや食材・単価を追加して、提案できる範囲を広げよう。追加したら `check_recipe_data()` で確認する | `RECIPES`（2-4）、`PRICE_TABLE`（2-2）、`INGREDIENT_ALIASES`（2-3） |
| 4 | Pythonの処理（表示） | 材料費の内訳を高い順に並べたり、合計に対する割合（%）を表示したりしてみよう | `show_candidate`（3-5） |
| 5 | Pythonの処理 | 候補がないとき、どの条件を「おまかせ」にすると何品見つかるかを表示してみよう（結果を自動で表示し直すのではなく、見直しの参考として示す） | `show_no_candidates`（3-5）、`suggest_recipes`（3-4） |
| 6 | AIへの指示 | 生成AIモードの指示文に「15分以内で作れる料理を優先する」などを加えて、提案の変化を登録済みレシピの提案と比べよう。Python の確認処理は変えずに比べる | `build_ai_prompt`（6-2） |

発展課題の例：入力欄やボタンを付ける（ipywidgets など）、複数の料理を組み合わせた献立を考える、Webアプリにする、登録されていない食材を生成AIモードで扱う（単価の扱い方を決める必要があります）。

試した条件、結果、気付いた点、改善案、改善後の確認結果は、リポジトリの `projects/recipe-suggester/docs/experiments.md` の表に記録しましょう。

## 6. 生成AIによる提案（任意の追加機能）

この節は、2〜5節の基本モードとは独立した追加機能です。**初期状態では無効（`USE_GEMINI = False`）で、APIキーを設定しなくても、ノートブック全体を最後までエラーなく実行できます。**

有効にすると、Google の Gemini API に入力条件を送り、料理の候補を JSON（決まった形のデータ）で受け取ります。受け取った候補はそのまま使わず、Python で次の点を確かめてから「生成AIによる提案」として表示します。

- 材料が、単価表の食材IDと単位で書かれているか（知らない食材ID、数でない量、0以下の量は採用しません）
- 分類と「必ず使う食材」が条件に合っているか（基本モードと同じ `unmet_conditions` で確かめます）
- 予算内か（費用は基本モードと同じ `calculate_cost` で計算し、AIが見積もった金額は使いません）

制約と注意：

- 費用を計算できるように、初版の生成AIモードでは、単価表に登録された食材だけを使わせます。登録されていない食材を扱う機能は発展課題です。
- 生成された分量、手順、所要時間は正しいとは限りません。確認してから使ってください。アレルギーへの対応や食の安全を保証する機能ではありません。
- APIを使えないとき（キーが未設定、利用回数の上限、通信の失敗など）や、条件を満たす候補がないときは、理由を表示して「登録済みレシピからの提案」に切り替えます。
- 1回の実行で API に送る問い合わせは1回です。利用制限やサーバーの一時的なエラーのときに SDK が自動で再送するのは、多くても1回です。
- 無料枠では、送った内容が Google のサービス改善に使われることがあります。個人情報は入力しないでください。

### 使うための準備（有効にする場合だけ）

1. [Google AI Studio](https://aistudio.google.com/apikey) で Gemini API の APIキーを作成します。無料枠の範囲で使う場合、課金の設定は不要です（無料枠の上限は AI Studio で確認できます）。
2. Colab の左側にある鍵のアイコン（シークレット）を開き、名前を `GEMINI_API_KEY`、値を APIキーとして登録し、「ノートブックからのアクセス」をオンにします。
3. 下の設定セルで `USE_GEMINI = True` に変えて、6節のセルを上から順に実行します。

APIキーをノートブックやコードに直接書かないでください。共有や GitHub への登録で漏れるおそれがあります。

### 6-1. 設定

In [ ]:
# ===== 生成AIモードの設定 =====
USE_GEMINI = False  # ★True にしたときだけ Gemini API を使います（初期状態は False で、APIを呼びません）

# 使うモデルの名前です。変更するときは、ここだけを書き換えます。
# 2026年10月28日に公式ドキュメント（URL は README に記載）で確認した、無料枠のある安定版のモデルです。
# 利用回数の上限にかかりやすいときは "gemini-3.5-flash-lite" なども試せます。
GEMINI_MODEL = "gemini-3.8-flash"

### 6-2. AIへの指示（プロンプト）と、受け取るデータの形

`build_ai_prompt` は、入力条件と使える食材の一覧から、AIへの指示文を作ります。`AI_RESPONSE_SCHEMA` は、AIに返してほしい JSON の形です。

**★ここを変えて試そう（AIへの指示を変える改善）**：`build_ai_prompt` の文章を変えると、提案の傾向が変わります。指示を変えても、受け取った後の確認（6-3）と費用の計算は、Python が同じように行います。

下の2つ目のセルを実行すると、4節の `conditions` で作った指示文を確認できます（APIは呼びません）。

In [ ]:
# 生成AIに返してほしい JSON の形（JSON Schema）。項目は RECIPES のレシピとほぼ同じです。
# 材料だけは [{"food_id": ..., "amount": ...}, ...] というリストで受け取り、確認するときに辞書に直します。
AI_RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "recipes": {
            "type": "array",
            "maxItems": MAX_SUGGESTIONS,
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "description": "料理名"},
                    "dish_types": {"type": "array", "items": {"type": "string", "enum": CHOICES["dish_type"]}},
                    "main_categories": {"type": "array", "items": {"type": "string", "enum": CHOICES["main_category"]}},
                    "genre": {"type": "string", "enum": CHOICES["genre"]},
                    "meals": {"type": "array", "items": {"type": "string", "enum": CHOICES["meal"]}},
                    "base_servings": {"type": "integer", "description": "材料の量が何人分か"},
                    "ingredients": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "food_id": {"type": "string", "description": "使える食材の一覧にある食材ID"},
                                "amount": {"type": "number", "description": "基準人数分の使用量（一覧の単位）"},
                            },
                            "required": ["food_id", "amount"],
                        },
                    },
                    "steps": {"type": "array", "items": {"type": "string"}, "description": "作り方（分量は書かない）"},
                    "minutes": {"type": "integer", "description": "目安の所要時間（分）"},
                },
                "required": ["name", "dish_types", "main_categories", "genre", "meals",
                             "base_servings", "ingredients", "steps", "minutes"],
            },
        },
    },
    "required": ["recipes"],
}


def ids_text_for_ai(food_ids):
    """食材IDのリストを「卵（egg）、長ねぎ（spring_onion）」のような文字列にする。"""
    if len(food_ids) == 0:
        return "なし"
    parts = []
    for food_id in food_ids:
        parts.append(food_name(food_id) + "（" + food_id + "）")
    return "、".join(parts)


def build_ai_prompt(request):
    """生成AIへの指示文（プロンプト）を作る。★ここを変えて試そう（AIへの指示を変える改善）"""
    lines = []
    lines.append("あなたは家庭料理のレシピを考えるアシスタントです。")
    lines.append("次の条件に合う家庭料理を、" + str(MAX_SUGGESTIONS) + "品まで提案してください。1品ずつ独立した料理にしてください。")
    lines.append("")
    lines.append("# 条件")
    for key in CHOICES:
        if request[key] == "おまかせ":
            lines.append("- " + LABELS[key] + "：指定なし")
        else:
            lines.append("- " + LABELS[key] + "：" + request[key])
    lines.append("- 基準人数（base_servings）：" + str(request["servings"]) + "人分で書く")
    if request["budget_per_person"] is None:
        lines.append("- 予算：指定なし")
    else:
        lines.append("- 予算：材料費が1人当たり" + format_amount(request["budget_per_person"])
                     + "円以下（下の参考単価で計算する）")
    lines.append("- 必ず使う食材（すべて使う）：" + ids_text_for_ai(request["required_ids"]))
    lines.append("- できれば使いたい食材：" + ids_text_for_ai(request["wanted_ids"]))
    lines.append("")
    lines.append("# 守ること")
    lines.append("- 材料には、下の「使える食材」にある食材ID（food_id）だけを使う。一覧にない食材や調味料は使わない。")
    lines.append("- amount は基準人数分の使用量で、一覧の単位で書いた0より大きい数にする。")
    lines.append("- dish_types、main_categories、genre、meals は、上の条件に合うように選ぶ。")
    lines.append("- steps には分量を書かない（人数に合わせて材料の量を計算し直すため）。ゆでるための湯は材料に含めない。")
    lines.append("- 価格や費用は出力しない（費用はプログラムが計算する）。")
    lines.append("")
    lines.append("# 使える食材（食材ID：名前、単位、参考単価）")
    for food_id in PRICE_TABLE:
        if PRICE_TABLE[food_id]["price"] is not None:  # 単価が分からない食材は使わせない
            lines.append("- " + food_id + "：" + PRICE_TABLE[food_id]["name"] + "、"
                         + PRICE_TABLE[food_id]["unit"] + "、" + price_text(food_id))
    return "\n".join(lines)

In [ ]:
# 4節の conditions で作った指示文を表示する（APIは呼びません）
preview_request, preview_errors, preview_notes = check_conditions(conditions)
if preview_request is None:
    print("4節の入力条件に直すところがあるため、指示文を作れません。")
else:
    print(build_ai_prompt(preview_request))

### 6-3. 受け取った提案を Python で確かめる

AIの応答は、JSON の文字列として受け取ります。`check_ai_recipes` は、それを読み取って1品ずつ次の順に確かめ、採用する候補と採用しない候補に分けます。

1. 形式：必要な項目がそろっているか、食材IDが単価表にあるか、量が0より大きい数か（`convert_ai_recipe` と 3-6 の `find_recipe_problems`）
2. 条件：分類と「必ず使う食材」が合っているか（3-2 の `unmet_conditions`）
3. 予算：基本モードと同じ `calculate_cost` で計算して、予算内か

採用した候補は、基本モードと同じ `ranking_score` で並べます。

In [ ]:
import json

AI_NOTE = (
    "※ 生成AIの提案について：材料は単価表の食材から選ばせ、費用は単価表から Python で計算しています（AIの見積もりは使っていません）。\n"
    "  分量・手順・所要時間は正しいとは限りません。確認してから使ってください。アレルギーや食の安全を確かめる機能ではありません。"
)


def convert_ai_recipe(item, number):
    """AIが返した1品分のデータを、RECIPES のレシピと同じ形の辞書に直す。
    戻り値は (recipe, problems)。problems は形式の問題のリスト（空なら問題なし）。"""
    if not isinstance(item, dict):
        return None, ["料理のデータが JSON のオブジェクトになっていません"]
    problems = []
    ingredients = {}
    raw_ingredients = item.get("ingredients")
    if not isinstance(raw_ingredients, list):
        raw_ingredients = []  # 材料がない問題は、下の find_recipe_problems が知らせる
    for entry in raw_ingredients:
        if not isinstance(entry, dict):
            problems.append("材料の書き方が正しくありません（今の値：" + repr(entry) + "）")
            continue
        food_id = entry.get("food_id")
        if not isinstance(food_id, str):
            problems.append("食材ID（food_id）が文字列ではありません（今の値：" + repr(food_id) + "）")
            continue
        if food_id in ingredients:
            problems.append("食材ID「" + food_id + "」が2回以上出てきます")
            continue
        ingredients[food_id] = entry.get("amount")

    recipe = {
        "id": "AI" + str(number),
        "name": item.get("name"),
        "dish_types": item.get("dish_types"),
        "main_categories": item.get("main_categories"),
        "genre": item.get("genre"),
        "meals": item.get("meals"),
        "base_servings": item.get("base_servings"),
        "ingredients": ingredients,
        "steps": item.get("steps"),
        "minutes": item.get("minutes"),
    }
    problems = problems + find_recipe_problems(recipe)
    return recipe, problems


def check_ai_recipes(response_text, request):
    """AIの応答（JSON の文字列）を読み取り、1品ずつ確かめる。
    戻り値の辞書：
      "accepted" … 採用する候補のリスト（基本モードの候補と同じ形で、同じ順位付けをする）
      "rejected" … 採用しない料理の {"name", "problems"} のリスト
      "error"    … 応答全体を読み取れなかった理由（読み取れたら None）
    """
    result = {"accepted": [], "rejected": [], "error": None}
    try:
        data = json.loads(response_text)
    except (TypeError, ValueError):
        result["error"] = "AIの応答を JSON として読み取れませんでした"
        return result
    if not isinstance(data, dict) or not isinstance(data.get("recipes"), list):
        result["error"] = "AIの応答に料理のリスト（recipes）がありません"
        return result

    number = 1
    for item in data["recipes"]:
        recipe, problems = convert_ai_recipe(item, number)
        number = number + 1
        cost = None
        if len(problems) == 0:
            # 形式に問題がなければ、基本モードと同じ関数で条件と予算を確かめる
            problems = unmet_conditions(recipe, request)
            cost = calculate_cost(recipe, request["servings"])
            budget = request["budget_per_person"]
            if budget is not None:
                if cost["per_person"] is None:
                    problems.append("費用を計算できないため、予算内か判定できません")
                elif cost["per_person"] > budget:
                    problems.append("1人当たりの予算（" + format_amount(budget) + "円）を超えています（"
                                    + format_yen(cost["per_person"]) + "）")
        if len(problems) > 0:
            name = "（料理名なし）"
            if isinstance(item, dict) and isinstance(item.get("name"), str):
                name = item["name"]
            result["rejected"].append({"name": name, "problems": problems})
        else:
            result["accepted"].append({
                "recipe": recipe,
                "cost": cost,
                "wanted_count": count_wanted(recipe, request["wanted_ids"]),
            })

    ranked = sorted(result["accepted"], key=ranking_score, reverse=True)
    result["accepted"] = ranked[:MAX_SUGGESTIONS]
    return result


def show_ai_result(result, request, label, note):
    """check_ai_recipes の結果を表示する。label は候補の見出しに付ける説明、note は注意書き。"""
    if len(result["rejected"]) > 0:
        print("【採用しなかった候補】Python で確かめたところ、次の問題がありました。")
        for item in result["rejected"]:
            print("・" + item["name"])
            for problem in item["problems"]:
                print("    - " + problem)
        print()
    if len(result["accepted"]) == 0:
        print("採用できる候補はありませんでした。")
        return
    print("採用した候補：" + str(len(result["accepted"])) + "品")
    print(PRICE_NOTE)
    print(note)
    number = 1
    for candidate in result["accepted"]:
        show_candidate(number, candidate, request, label)
        number = number + 1

### 6-4. 確認用サンプルで、確かめる仕組みを試す（APIキー不要）

AIの応答と同じ形の JSON を手作業で作り、6-3 の確認処理に通します。**これは AI の出力ではありません。** わざと問題のある候補（単価表にない食材ID、マイナスの量、必ず使う食材を使っていない、予算を超える）を入れて、Python の確認で見つけられるかを試します。

In [ ]:
# 確認用サンプル：AIの応答と同じ形の JSON を手作業で作ったものです（AIの出力ではありません）。
SAMPLE_AI_RESPONSE = """
{
  "recipes": [
    {
      "name": "卵とじうどん",
      "dish_types": ["麺類"],
      "main_categories": ["卵・乳製品"],
      "genre": "和食",
      "meals": ["昼食", "夕食"],
      "base_servings": 2,
      "ingredients": [
        {"food_id": "udon", "amount": 2},
        {"food_id": "egg", "amount": 2},
        {"food_id": "spring_onion", "amount": 0.5},
        {"food_id": "water", "amount": 600},
        {"food_id": "dashi_powder", "amount": 4},
        {"food_id": "soy_sauce", "amount": 30},
        {"food_id": "potato_starch", "amount": 4}
      ],
      "steps": [
        "長ねぎを斜め薄切りにし、卵を溶きほぐす。片栗粉は、分量のうち少しの水で溶いておく。",
        "鍋に残りの水と和風だしの素を入れて沸かし、しょうゆで味を調える。",
        "ゆでうどんと長ねぎを加えて温め、水で溶いた片栗粉でとろみをつける。",
        "溶き卵を回し入れ、ふんわり固まったら火を止める。"
      ],
      "minutes": 15
    },
    {
      "name": "しいたけと卵のすまし汁",
      "dish_types": ["スープ"],
      "main_categories": ["野菜・豆類"],
      "genre": "和食",
      "meals": ["夕食"],
      "base_servings": 2,
      "ingredients": [
        {"food_id": "shiitake", "amount": 4},
        {"food_id": "egg", "amount": -1},
        {"food_id": "water", "amount": 400},
        {"food_id": "dashi_powder", "amount": 2},
        {"food_id": "soy_sauce", "amount": 10}
      ],
      "steps": [
        "しいたけを薄切りにする。",
        "鍋に水と和風だしの素を入れて沸かし、しいたけを煮る。",
        "しょうゆで味を調え、溶き卵を回し入れる。"
      ],
      "minutes": 10
    },
    {
      "name": "鶏の照り焼き",
      "dish_types": ["おかず"],
      "main_categories": ["肉"],
      "genre": "和食",
      "meals": ["夕食"],
      "base_servings": 2,
      "ingredients": [
        {"food_id": "chicken", "amount": 500},
        {"food_id": "soy_sauce", "amount": 30},
        {"food_id": "sugar", "amount": 15},
        {"food_id": "oil", "amount": 10}
      ],
      "steps": [
        "鶏もも肉の厚い部分を開き、しょうゆと砂糖を混ぜてたれを作る。",
        "フライパンにサラダ油を熱し、鶏もも肉を皮の面から焼く。",
        "両面が焼けたらたれを加え、照りが出るまで煮からめる。"
      ],
      "minutes": 20
    }
  ]
}
"""

# サンプルを確かめるときの条件（4節の conditions とは別に決めています）
sample_conditions = {
    "dish_type": "おまかせ",
    "main_category": "おまかせ",
    "genre": "和食",
    "meal": "夕食",
    "servings": 2,
    "budget_per_person": 300,
    "required_ingredients": ["卵"],
    "wanted_ingredients": ["長ねぎ"],
}

print("【確認用サンプル】手作業で作ったデータです。AIの出力ではなく、APIも呼んでいません。")
sample_request = prepare_request(sample_conditions)
print()
sample_result = check_ai_recipes(SAMPLE_AI_RESPONSE, sample_request)
show_ai_result(sample_result, sample_request, "確認用サンプル・AIの出力ではありません",
               "※ 確認の仕組みを試すための手作りのデータです。")

### 6-5. Gemini API への接続

ここから下は、Gemini API に固有の処理です。`USE_GEMINI = True` のときだけ使われ、無効のときはパッケージの導入もAPIキーの読み込みもしません。

- `prepare_gemini_sdk`：公式の Python SDK `google-genai`（2.3.0 以降）が無いか古いときだけ、pip で導入します。
- `read_gemini_api_key`：Colab では「シークレット」の `GEMINI_API_KEY` を読み込みます（Colab 以外で試すときは、同じ名前の環境変数を使います）。キーの値は表示しません。
- `ask_gemini`：Interactions API（`client.interactions.create`）に1回だけ問い合わせ、決めた JSON の形で応答を受け取ります。失敗したときは、短い説明を返します。

In [ ]:
import importlib
import importlib.metadata
import os
import subprocess
import sys

GEMINI_SDK_REQUIREMENT = "google-genai>=2.3.0"  # Interactions API を使える版


def gemini_sdk_ready():
    """google-genai の 2.3.0 以降が入っていれば True を返す（ここでは import しません）。"""
    try:
        version = importlib.metadata.version("google-genai")
    except importlib.metadata.PackageNotFoundError:
        return False
    parts = version.split(".")
    try:
        major = int(parts[0])
        minor = int(parts[1])
    except (IndexError, ValueError):
        return False
    return major > 2 or (major == 2 and minor >= 3)


def prepare_gemini_sdk():
    """google-genai を使える状態にする。入っていないか古いときだけ、pip で導入する。"""
    if gemini_sdk_ready():
        return True
    print("Gemini API 用のパッケージ google-genai を導入します（1分ほどかかることがあります）。")
    command = [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", GEMINI_SDK_REQUIREMENT]
    completed = subprocess.run(command, capture_output=True, text=True)
    importlib.invalidate_caches()
    if completed.returncode != 0 or not gemini_sdk_ready():
        print("google-genai を導入できませんでした。ネットワークの状態を確認してください。")
        return False
    return True


def read_gemini_api_key():
    """APIキーを読み込む。Colab では「シークレット」の GEMINI_API_KEY、Colab 以外では同じ名前の環境変数を使う。
    キーの値は表示しない。読み込めないときは理由を表示して None を返す。"""
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None  # Colab 以外で実行している

    if userdata is None:
        api_key = os.environ.get("GEMINI_API_KEY")
        place = "環境変数 GEMINI_API_KEY"
    else:
        place = "Colab のシークレット GEMINI_API_KEY"
        try:
            api_key = userdata.get("GEMINI_API_KEY")
        except userdata.SecretNotFoundError:
            api_key = None
        except userdata.NotebookAccessError:
            print("このノートブックには、シークレット GEMINI_API_KEY を使う許可がありません。")
            print("左側の鍵のアイコンを開き、GEMINI_API_KEY の「ノートブックからのアクセス」をオンにしてください。")
            return None
        except Exception as error:
            print("シークレットを読み込めませんでした（" + type(error).__name__ + "）。Colab の画面から実行してください。")
            return None

    if not isinstance(api_key, str) or api_key.strip() == "":
        print(place + " が設定されていません。6節の「使うための準備」を確認してください。")
        return None
    return api_key.strip()


def explain_gemini_error(error):
    """API を呼んだときの失敗を、短い説明にする（キーや送った内容は表示しない）。"""
    status = getattr(error, "status_code", None)  # HTTP の状態コード（無いときは None）
    name = type(error).__name__
    if status == 400:
        return "送った内容が受け付けられませんでした（400）。GEMINI_MODEL などの設定を確認してください"
    if status == 401 or status == 403:
        return "APIキーが無効か、使う権限がありません（" + str(status) + "）"
    if status == 404:
        return "モデルが見つかりません（404）。GEMINI_MODEL のモデル名を確認してください"
    if status == 429:
        return "利用回数の上限（無料枠など）に達しました（429）。時間をおいてから試してください"
    if isinstance(status, int) and status >= 500:
        return "Gemini API 側で一時的な問題が起きています（" + str(status) + "）。時間をおいてから試してください"
    if "Timeout" in name:
        return "時間内に応答がありませんでした（タイムアウト）"
    if "Connection" in name:
        return "Gemini API に接続できませんでした。ネットワークの状態を確認してください"
    if isinstance(error, (ImportError, AttributeError)):
        return "google-genai を正しく読み込めませんでした。「ランタイム」→「セッションを再起動」してから、もう一度実行してください"
    return "予期しないエラーが起きました（" + name + "）"


def ask_gemini(prompt):
    """Gemini API に1回だけ問い合わせて、応答の文字列を返す。
    戻り値は (応答の文字列, 失敗の理由)。成功したときの失敗の理由は None。"""
    if not prepare_gemini_sdk():
        return None, "Gemini API 用のパッケージ google-genai を準備できませんでした"
    api_key = read_gemini_api_key()
    if api_key is None:
        return None, "APIキーを読み込めませんでした"
    try:
        from google import genai

        client = genai.Client(
            api_key=api_key,
            # 90秒で打ち切り、SDK の自動再試行を最小にする（429 や 5xx のときの再送は多くても1回）
            http_options={"timeout": 90000, "retry_options": {"attempts": 1}},
        )
        interaction = client.interactions.create(
            model=GEMINI_MODEL,
            input=prompt,
            response_format={
                "type": "text",
                "mime_type": "application/json",
                "schema": AI_RESPONSE_SCHEMA,
            },
            store=False,  # 1回だけの問い合わせなので、Google 側に会話を保存しない
        )
    except Exception as error:
        return None, explain_gemini_error(error)
    text = getattr(interaction, "output_text", None)
    if not isinstance(text, str) or text.strip() == "":
        return None, "生成AIから文章の応答がありませんでした"
    return text, None

### 6-6. 生成AIモードの実行

`run_ai_mode` は、入力の確認 → AIへの問い合わせ（1回） → Python での確認 → 表示 の順に進みます。AIの提案を表示できないときは、理由を表示してから「登録済みレシピからの提案」に切り替えます。

In [ ]:
def switch_to_basic(reason, request):
    """生成AIによる提案を出せないときに、理由を表示して基本モードに切り替える。"""
    print()
    print("【切替】生成AIによる提案を表示できません。")
    print("理由：" + reason + "。")
    print("登録済みレシピからの提案に切り替えます。")
    show_basic_result(request)


def run_ai_mode(conditions):
    """生成AIモード。USE_GEMINI が True のときだけ、Gemini API に1回問い合わせる。"""
    if not USE_GEMINI:
        print("生成AIモードは無効です（USE_GEMINI = False）。APIは呼んでいません。")
        print("登録済みレシピからの提案は、4節のセルで表示できます。")
        return
    print("【生成AIモード】モデル：" + GEMINI_MODEL)
    request = prepare_request(conditions)
    if request is None:
        print("入力条件を直してから、もう一度実行してください（生成AIには送っていません）。")
        return

    reason = None
    result = None
    try:
        text, reason = ask_gemini(build_ai_prompt(request))
        if reason is None:
            result = check_ai_recipes(text, request)
            reason = result["error"]
    except Exception as error:  # 思わぬエラーが起きても、基本モードは使えるようにする
        reason = "予期しないエラー（" + type(error).__name__ + "）"
    if reason is not None:
        switch_to_basic(reason, request)
        return

    print()
    show_ai_result(result, request, "生成AIによる提案", AI_NOTE)
    if len(result["accepted"]) == 0:
        switch_to_basic("条件をすべて満たす候補がありませんでした", request)

In [ ]:
# 生成AIモードを実行する（4節の conditions を使います）。USE_GEMINI = False のときは、APIを呼ばずに終わります。
run_ai_mode(conditions)